# Эксперимент на реальных данных: MovieLens-20M

## Цель

Проверить, воспроизводятся ли гипотезы **H1** (рост KL-дивергенции) 
и **H3** (коллапс tr(Σ̂^u)) на **реальных** пользовательских данных, 
а не только в синтетической симуляции.

## Подход

1. **Данные**: MovieLens-20M — 20 млн оценок (1–5 звёзд) от 138K пользователей для 27K фильмов.
2. **Фильтрация**: оставляем только активных пользователей (≥50 оценок) и популярные фильмы (≥100 оценок), затем берём топ-500 × топ-500.
3. **Эмбеддинги**: матричная факторизация (d=16) -> векторы пользователей и фильмов.
4. **Симуляция**: `EmpiricalUserGenerator` + `SimulationEnvironment` с двумя режимами.
5. **Проверка H1, H3**: сравниваем `closed_loop` и `no_influence`.

## Зачем нужен реальный датасет?

Синтетические эксперименты (H1–H6) работают с GMM-пользователями и случайными объектами. 
Реальные данные проверяют, что:
- Исходные пользовательские паттерны (кластеризация вкусов) сохраняются в эмбеддингах MF.
- Feedback loop проявляется при **настоящей** структуре предпочтений, а не артефактной.
- Все наши численные оценки KL и tr(Σ) имеют смысл за пределами синтетики.

## Ключевые термины

| Термин | Смысл |
|--------|-------|
| **Rating matrix** | Матрица R[u, i] = оценка пользователя u для фильма i (NaN если не оценён) |
| **Матричная факторизация (MF)** | Разложение R ≈ U · Vᵀ, где U — эмбеддинги пользователей, V — фильмов |
| **EmpiricalUserGenerator** | Генератор новых пользователей, сэмплирующий из N(μ_t, Σ_t) текущей популяции |
| **KL-дивергенция** | Расстояние между текущим и начальным распределением пользователей |
| **tr(Σ̂^u)** | Суммарная дисперсия эмбеддингов активных пользователей |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader

from sim.environment import SimulationEnvironment, ExperimentDataset
from sim.user_generator import EmpiricalUserGenerator
from sim.click_model import ClickModel
from models.rec_models import RecModel
from models.serving import ServingPolicy

FIGURES_DIR = Path('../paper/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVE_DIR = Path('../../data/movielens')
print('Imports OK')


Imports OK


In [2]:
# ── 1. Загрузка и фильтрация данных ──────────────────────────────────
print('Loading ratings...', flush=True)
ratings = pd.read_csv(ARCHIVE_DIR / 'rating.csv')
ratings.columns = ['userId', 'movieId', 'rating', 'timestamp']

# Фильтр: пользователи ≥50 оценок, фильмы ≥100 оценок
user_counts = ratings.groupby('userId').size()
movie_counts = ratings.groupby('movieId').size()

active_users = user_counts[user_counts >= 50].index
popular_movies = movie_counts[movie_counts >= 100].index

ratings = ratings[ratings['userId'].isin(active_users) &
 ratings['movieId'].isin(popular_movies)].copy()
print(f'After filter: {ratings.userId.nunique()} users, {ratings.movieId.nunique()} movies, '
 f'{len(ratings):,} ratings')

# Ограничиваем до N_USERS самых активных пользователей и N_MOVIES популярных
N_USERS = 500
N_MOVIES = 500

top_users = user_counts.reindex(active_users).nlargest(N_USERS).index
top_movies = movie_counts.reindex(popular_movies).nlargest(N_MOVIES).index

ratings = ratings[ratings['userId'].isin(top_users) &
 ratings['movieId'].isin(top_movies)].copy()

# Переиндексируем
user_map = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}
movie_map = {m: i for i, m in enumerate(sorted(ratings.movieId.unique()))}
ratings['uid'] = ratings['userId'].map(user_map)
ratings['mid'] = ratings['movieId'].map(movie_map)

n_users = ratings['uid'].nunique()
n_movies = ratings['mid'].nunique()

print(f'Working subset: {n_users} users, {n_movies} movies, {len(ratings):,} ratings')

# Матрица оценок (бинаризованная: ≥3.5 -> позитивная)
rating_matrix = np.full((n_users, n_movies), np.nan)
for _, row in ratings.iterrows():
 rating_matrix[int(row['uid']), int(row['mid'])] = float(row['rating'] >= 3.5)
print(f'Rating matrix: {np.sum(~np.isnan(rating_matrix)):,} known entries, '
 f'{np.nanmean(rating_matrix):.2%} positive')


Loading ratings...


After filter: 85307 users, 8546 movies, 18,049,622 ratings


Working subset: 500 users, 500 movies, 193,401 ratings


Rating matrix: 193,401 known entries, 61.31% positive


## Шаг 1: Загрузка и фильтрация MovieLens-20M

### Датасет

**MovieLens-20M** — стандартный бенчмарк в области рекомендательных систем:
- 20 млн оценок от 138 493 пользователей для 27 278 фильмов.
- Оценки: от 0.5 до 5.0 с шагом 0.5.
- Временной диапазон: 1995–2015.
- Источник: GroupLens Research, University of Minnesota.

### Протокол фильтрации

**Проблема «холодного старта»**: пользователи с малым числом оценок и редкие фильмы 
дают зашумлённые эмбеддинги. Фильтруем:

| Условие | Порог | Смысл |
|---------|-------|-------|
| `user_counts >= 50` | ≥50 оценок | Активный пользователь с достаточной историей |
| `movie_counts >= 100` | ≥100 оценок | Популярный фильм с достаточной статистикой |
| Топ-500 × 500 | По числу оценок | Ограничение для управляемого эксперимента |

После фильтрации: **500 пользователей × 500 фильмов**, плотность матрицы ~2–5%.

In [3]:
# ── 2. Матричная факторизация (ALS-like с SGD) ───────────────────────
EMB_DIM = 16
EPOCHS = 30
LR = 0.01
BATCH = 2048

# Формируем тренировочные пары из известных записей
known = np.argwhere(~np.isnan(rating_matrix))
labels = rating_matrix[known[:, 0], known[:, 1]].astype(np.float32)

uids_t = torch.tensor(known[:, 0], dtype=torch.long)
mids_t = torch.tensor(known[:, 1], dtype=torch.long)
labels_t = torch.tensor(labels)

class MatrixFactorization(nn.Module):
 def __init__(self, n_users, n_items, emb_dim):
 super().__init__()
 self.user_emb = nn.Embedding(n_users, emb_dim)
 self.item_emb = nn.Embedding(n_items, emb_dim)
 nn.init.normal_(self.user_emb.weight, std=0.1)
 nn.init.normal_(self.item_emb.weight, std=0.1)

 def forward(self, u, i):
 return torch.sigmoid((self.user_emb(u) * self.item_emb(i)).sum(dim=-1))

mf = MatrixFactorization(n_users, n_movies, EMB_DIM)
optimizer = optim.Adam(mf.parameters(), lr=LR)
criterion = nn.BCELoss()

dataset_t = TensorDataset(uids_t, mids_t, labels_t)
loader = DataLoader(dataset_t, batch_size=BATCH, shuffle=True)

print(f'Training MF: {n_users}×{n_movies}, d={EMB_DIM}, {EPOCHS} epochs...')
for epoch in range(EPOCHS):
 total_loss = 0.0
 for u_b, i_b, lbl_b in loader:
 optimizer.zero_grad()
 pred = mf(u_b, i_b)
 loss = criterion(pred, lbl_b)
 loss.backward()
 optimizer.step()
 total_loss += loss.item()
 if (epoch + 1) % 5 == 0:
 print(f' Epoch {epoch+1:2d}: loss={total_loss/len(loader):.4f}')

user_emb_np = mf.user_emb.weight.detach().numpy().astype(np.float64)
item_emb_np = mf.item_emb.weight.detach().numpy().astype(np.float64)
print(f'Embeddings: users {user_emb_np.shape}, items {item_emb_np.shape}')
print(f'User emb norm (mean): {np.linalg.norm(user_emb_np, axis=1).mean():.3f}')


Training MF: 500×500, d=16, 30 epochs...


 Epoch 5: loss=0.4448


 Epoch 10: loss=0.4107


 Epoch 15: loss=0.4016


 Epoch 20: loss=0.3974


 Epoch 25: loss=0.3948


 Epoch 30: loss=0.3930
Embeddings: users (500, 16), items (500, 16)
User emb norm (mean): 2.945


## Шаг 2: Матричная факторизация (SGD-based MF)

### Идея

Матрица оценок R ∈ ℝ^{N×M} (N пользователей, M фильмов) неполная (sparse). 
Ищем низкоранговое приближение:
$$R \approx U V^T, \quad U \in \mathbb{R}^{N \times d},\; V \in \mathbb{R}^{M \times d}$$

Строки U — **эмбеддинги пользователей** (их «вкусовой профиль»), 
строки V — **эмбеддинги фильмов** (их «содержательный профиль»).

### Реализация

`MatrixFactorization` — нейросеть PyTorch:
- `nn.Embedding(N, d)` — таблица эмбеддингов пользователей.
- `nn.Embedding(M, d)` — таблица эмбеддингов фильмов.
- Forward: `score(u, i) = sigmoid(U[u] · V[i])`.

Обучение: SGD с `BCELoss` на известных парах (u, i).

| Параметр | Значение |
|----------|---------|
| `EMB_DIM` d | 16 |
| `EPOCHS` | 30 |
| `LR` | 0.01 |
| `BATCH` | 2048 |

### Допущение о представимости

Предполагаем, что 30 эпох обучения достаточно для выявления основных кластеров вкусов 
(жанровые предпочтения, пр.). В реальных системах используются более продвинутые 
методы (ALS, BPR, нейросетевые факторизации), но для нашей цели MF достаточно: 
нам важна **структура** эмбеддингов, а не точность предсказаний.

In [4]:
# ── 3. Запуск симуляции ──────────────────────────────────────────────
T = 80
T_RET = 10
K_REC = 10
USER_DRIFT = 0.005
ADHERENCE = 0.7
N_SEEDS = 3

MODES = ['closed_loop', 'no_influence']
COLORS = {'closed_loop': '#d62728', 'no_influence': '#1f77b4'}
LABELS = {'closed_loop': f'closed\\_loop (α={ADHERENCE})',
 'no_influence': r'no\\_influence (α=0)'}

def build_real_env(mode, seed):
 np.random.seed(seed); torch.manual_seed(seed)
 
 # Перемешиваем начальный порядок пользователей
 rng = np.random.default_rng(seed)
 idx = rng.choice(n_users, n_users, replace=False)
 
 u_emb = user_emb_np[idx].copy()
 i_emb = item_emb_np.copy()
 mat = rating_matrix[idx].copy()
 
 true_pref = mat.copy()
 true_pref = np.where(np.isnan(true_pref), 0.5, true_pref)
 
 dataset = ExperimentDataset(u_emb, i_emb, mat)
 gen = EmpiricalUserGenerator(replacement_rate=0.05, n_clusters=5, memory_effect=8)
 gen.initialize(u_emb)
 
 model = RecModel(EMB_DIM, EMB_DIM, hidden_size=64)
 policy = ServingPolicy('top_k')
 alpha_c = 0.0 if mode == 'no_influence' else ADHERENCE
 click = ClickModel(adherence=alpha_c, usage_rate=0.8)
 
 env = SimulationEnvironment(
 dataset=dataset, rec_model=model,
 user_generator=gen, click_model=click,
 serving_policy=policy, mode=mode,
 true_preference_matrix=true_pref,
 retrain_period=T_RET, K=K_REC,
 seen_filter=True,
 user_drift_beta=USER_DRIFT,
 drift_alpha=0.0)
 return env

results = {}
for mode in MODES:
 results[mode] = []
 for seed in range(N_SEEDS):
 env = build_real_env(mode, seed)
 for t in range(T):
 env.step(t)
 df = env.metrics.get_dataframe()
 results[mode].append(df)
 print(f' {mode:20s} seed={seed} '
 f'tr(Σ): {df["trace_sigma"].iloc[0]:.3f}->{df["trace_sigma"].iloc[-1]:.3f} '
 f'KL: {df["kl_from_initial"].iloc[-1]:.3f}')

print('All runs complete.')


 closed_loop seed=0 tr(Σ): 7.075->0.172 KL: 34.716


 closed_loop seed=1 tr(Σ): 7.087->0.175 KL: 34.599


 closed_loop seed=2 tr(Σ): 7.185->0.175 KL: 34.238


 no_influence seed=0 tr(Σ): 7.195->0.383 KL: 19.302


 no_influence seed=1 tr(Σ): 7.220->0.358 KL: 19.089


 no_influence seed=2 tr(Σ): 7.301->0.277 KL: 20.851
All runs complete.


## Шаг 3: Симуляция с EmpiricalUserGenerator

### Отличие от синтетики

В синтетических экспериментах (H1–H5) новые пользователи сэмплируются из GMM. 
Здесь пользователи генерируются из **эмпирического распределения** текущей популяции:
$$u_{\text{new}} \sim \mathcal{N}(\hat{\mu}_t, \hat{\Sigma}_t), \quad
\hat{\mu}_t = \frac{1}{N}\sum_i u_i^t, \quad
\hat{\Sigma}_t = \operatorname{Cov}(\{u_i^t\})$$

Это реалистичнее: новые пользователи похожи на тех, кто уже есть в системе.

### Параметры симуляции

| Параметр | Значение | Смысл |
|----------|---------|-------|
| T | 80 | Шагов симуляции |
| T_RET | 10 | Период переобучения (каждые 10 шагов) |
| K_REC | 10 | Размер рекомендательного списка |
| ADHERENCE α | 0.7 | Сила feedback loop |
| USER_DRIFT β | 0.005 | Скорость β-дрейфа эмбеддингов |
| N_SEEDS | 3 | Число независимых запусков |

### Два режима

| Режим | Что происходит |
|-------|---------------|
| `closed_loop` | Рекомендатель обучается на своих логах; α=0.7 -> сильный feedback loop |
| `no_influence` | α=0: пользователи кликают независимо от рекомендаций (базовая линия) |

In [5]:
# ── 4. Визуализация ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ts = results['closed_loop'][0]['t'].values

for mode in MODES:
 vals_sigma = np.array([df['trace_sigma'].values for df in results[mode]])
 mu_s, sd_s = vals_sigma.mean(0), vals_sigma.std(0)
 vals_kl = np.array([df['kl_from_initial'].values for df in results[mode]])
 mu_k, sd_k = vals_kl.mean(0), vals_kl.std(0)

 axes[0].plot(ts, mu_s, color=COLORS[mode], lw=2.5, label=LABELS[mode])
 axes[0].fill_between(ts, mu_s - sd_s, mu_s + sd_s, color=COLORS[mode], alpha=0.15)

 axes[1].plot(ts, mu_k, color=COLORS[mode], lw=2.5, label=LABELS[mode])
 axes[1].fill_between(ts, mu_k - sd_k, mu_k + sd_k, color=COLORS[mode], alpha=0.15)

axes[0].set_xlabel('Step t', fontsize=13)
axes[0].set_ylabel(r'$\mathrm{tr}(\hat{\Sigma}_t^u)$', fontsize=13)
axes[0].set_title('MovieLens: коллапс дисперсии', fontsize=13)
axes[0].legend(fontsize=10); axes[0].grid(True, linestyle='--', alpha=0.4)

axes[1].set_xlabel('Step t', fontsize=13)
axes[1].set_ylabel(r'$KL(P_t \| P_0)$', fontsize=13)
axes[1].set_title('MovieLens: дрейф от начального распределения', fontsize=13)
axes[1].legend(fontsize=10); axes[1].grid(True, linestyle='--', alpha=0.4)

fig.suptitle(f'Реальные данные MovieLens-20M (N={n_users}, d={EMB_DIM}, T={T}, α={ADHERENCE}, β={USER_DRIFT})',
 fontsize=10, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'real_data_movielens.pdf', bbox_inches='tight')
plt.show()
print('Saved: real_data_movielens.pdf')


Saved: real_data_movielens.pdf


In [6]:
# ── 5. Проверка гипотез на реальных данных ────────────────────────────
kl_cl = np.mean([df['kl_from_initial'].iloc[-1] for df in results['closed_loop']])
kl_ni = np.mean([df['kl_from_initial'].iloc[-1] for df in results['no_influence']])

sig_cl_0 = np.mean([df['trace_sigma'].iloc[0] for df in results['closed_loop']])
sig_cl_T = np.mean([df['trace_sigma'].iloc[-1] for df in results['closed_loop']])
sig_ni_0 = np.mean([df['trace_sigma'].iloc[0] for df in results['no_influence']])
sig_ni_T = np.mean([df['trace_sigma'].iloc[-1] for df in results['no_influence']])

drop_cl = (sig_cl_0 - sig_cl_T) / sig_cl_0 * 100
drop_ni = (sig_ni_0 - sig_ni_T) / sig_ni_0 * 100

print('=== MovieLens: итоговые метрики ===')
print(f' closed_loop: tr(Σ) {sig_cl_0:.3f}->{sig_cl_T:.3f} ({drop_cl:+.1f}%), KL={kl_cl:.3f}')
print(f' no_influence: tr(Σ) {sig_ni_0:.3f}->{sig_ni_T:.3f} ({drop_ni:+.1f}%), KL={kl_ni:.3f}')
print()
h1 = kl_cl > kl_ni
h3 = drop_cl > drop_ni
print(f'H1 (KL_cl > KL_ni): {" подтверждена" if h1 else " не подтверждена"} '
 f'({kl_cl:.3f} vs {kl_ni:.3f})')
print(f'H3 (drop_cl > drop_ni в Σ): {" подтверждена" if h3 else " не подтверждена"} '
 f'({drop_cl:.1f}% vs {drop_ni:.1f}%)')


=== MovieLens: итоговые метрики ===
 closed_loop: tr(Σ) 7.115->0.174 (+97.6%), KL=34.518
 no_influence: tr(Σ) 7.239->0.339 (+95.3%), KL=19.747

H1 (KL_cl > KL_ni): подтверждена (34.518 vs 19.747)
H3 (drop_cl > drop_ni в Σ): подтверждена (97.6% vs 95.3%)
